# Transformer-Inspired Analysis of Bilinear Compressive Security (BCS)

## Author Information
**Name:** Meraol Alemayehu  
**Nationality:** Ethiopian  
**Education:** MSc in Computer Science (Ongoing)  
**Undergraduate:** BSc in Information Technology, Arba Minch University  
**Affiliation:** Independent Researcher / Data Science & Cybersecurity Enthusiast  
**Email:** meralex3939@gmail.com  

---

## Project Type
Cybersecurity + Artificial Intelligence Research Project

---

## Domain Focus
- Cybersecurity Systems and Cryptographic Security Models  
- Artificial Intelligence (Machine Learning & Deep Learning)  
- Signal Processing and Compressive Sensing  
- Adversarial Machine Learning and Attack Simulation  

---

## Project Overview

This project investigates the security properties of **Bilinear Compressive Security (BCS)** systems based on the research paper by Flinth et al. The study simulates encrypted signal transmission where data is transformed using a secret linear key matrix and a random convolutional filter.

To analyze potential vulnerabilities, a transformer-inspired deep learning model is used to simulate an adversarial attacker attempting to extract structural information from observed ciphertext-like signals without direct access to the encryption key or filter.

The goal is to explore whether modern AI architectures can infer hidden statistical patterns in bilinear compressed representations and what this implies for the security assumptions of BCS systems.

---

## Objectives

- Simulate a Bilinear Compressive Security (BCS) encryption environment  
- Model encrypted signal generation using linear transformation + convolution  
- Apply transformer-based neural networks for adversarial inference  
- Analyze the potential leakage of structural information from repeated observations  
- Evaluate AI-based attack feasibility on compressive security systems  

---

## Research Contribution

This work provides an experimental AI-driven perspective on:

- The robustness of bilinear compressive security under repeated observations  
- The applicability of transformer architectures in cryptographic signal analysis  
- The intersection of machine learning and modern lightweight encryption schemes  

---

## Status
Experimental Research Prototype (Educational / Simulation Purpose)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

We simulate the paper’s model:

y=h∗(Qx)

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

n_samples = 3000
seq_len = 16
feature_dim = 12

# Secret key matrix Q (unknown to attacker)
Q = np.random.normal(0, 1, (feature_dim, feature_dim))

def generate_signal(label):
    x = np.random.normal(0, 1, (seq_len, feature_dim))

    # simulate sparsity differences
    if label == 1:
        x += np.random.normal(0.8, 0.5, (seq_len, feature_dim))

    # linear transformation (Qx)
    x = x @ Q

    # convolution-like filter h
    h = np.random.normal(0, 1, (seq_len, 1))
    y = x * h

    return y

X = []
y = []

for i in range(n_samples):
    label = np.random.randint(0, 2)
    X.append(generate_signal(label))
    y.append(label)

X = np.array(X)
y = np.array(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        pe = torch.zeros(50, d_model)
        position = torch.arange(0, 50).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class BCS_Attack_Transformer(nn.Module):
    def __init__(self, feature_dim, d_model=64):
        super().__init__()

        self.embed = nn.Linear(feature_dim, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.head(x)

In [ ]:
class BCS_Attack_Transformer(nn.Module):
    def __init__(self, feature_dim, d_model=64):
        super().__init__()

        self.embed = nn.Linear(feature_dim, d_model)
        self.pos = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.head(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BCS_Attack_Transformer(feature_dim=feature_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 20

for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_train.to(device))
    loss = criterion(outputs, y_train.to(device))

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:
model.eval()

with torch.no_grad():
    preds = model(X_test.to(device))
    preds = torch.argmax(preds, dim=1).cpu()

print("Accuracy:", accuracy_score(y_test, preds))

## Security Interpretation

This experiment simulates an adversarial setting based on Bilinear Compressive Security (BCS).

Key insights:

- The system introduces non-linear masking via convolution filters (h)
- Linear transformation via secret key matrix (Q)
- Attack model attempts to infer structure using transformer-based sequence learning

Findings:
- Even without direct access to Q or h, statistical learning models can extract partial structural information
- This supports the theoretical claim that repeated observations may leak information under certain conditions

This demonstrates a practical AI-based perspective on cryptanalytic leakage in BCS systems.